# Table detection: train longer, at higher resolution

The scaling study showed that the 18,000-page model **had not finished learning**:
at epoch 20, validation loss was still falling and mAP was still rising. Early
stopping never triggered — training ended only because it hit the epoch limit.

This notebook trains the same model on the **same 18,000 pages** with two changes:

| | Scaling study (18k run) | This notebook |
|---|---|---|
| Epochs | 20 | **60** |
| Early-stopping patience | 5 | **10** |
| Image size | 640 px | **1024 px** |
| Training / validation / test pages | 18,000 / 500 / 500 | identical pages |
| Model, batch, seed, augmentation | yolo11n, 8, 42, no mosaic/flips | unchanged |

**Why 1024 px?** DocLayNet pages are about 1025 px. At 640 px every page is shrunk
to ~40% of its area, which blurs thin table borders and small tables. 1024 px
keeps the page at roughly native resolution.

Note: two things change at once, so if the score improves we cannot say how much
came from each. That is a deliberate trade-off to save GPU time.

**Runtime:** roughly 4–6 hours on an RTX 4090, plus ~30–40 minutes of download.

## 1. Install

In [1]:
%pip install -q "ultralytics==8.3.221" "pyarrow==21.0.0" "huggingface_hub==0.36.0" pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [2]:
import json
import shutil
import time
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import torch
import yaml
from huggingface_hub import HfFileSystem
from PIL import Image
from ultralytics import YOLO, settings

## 3. Settings

Everything you might want to change is here.

In [3]:
# --- What this run changes ------------------------------------------------
EPOCHS = 60
PATIENCE = 10
IMGSZ = 1024

# --- Kept the same as the scaling study -----------------------------------
PRETRAINED = "yolo11n.pt"
BATCH = 8
SEED = 42
PAGES_TRAIN = 18000
PAGES_VALIDATION = 500
PAGES_TEST = 500
MAX_PAGES_PER_DOCUMENT = 10

# --- Data source -----------------------------------------------------------
REPO = "docling-project/DocLayNet-v1.2"
REVISION = "85cadd47c3ac59e1a2c3677ec72a02b143d96430"  # Pinned so the data never changes.
TABLE_CATEGORY_ID = 9  # "Table" in DocLayNet.
TABLE_CLASS_ID = 0     # "table" in our one-class YOLO dataset.

# --- Folders (change these if you run somewhere else) ----------------------
DATA = Path("/root/table_data")       # Downloaded pages and labels.
OUTPUT = Path("/root/long_training")  # Weights, plots and results.
WORKERS = 4                           # Dataloader processes.

# Optional: the page list saved by the scaling study. If this file exists we
# check that we picked exactly the same pages.
SCALING_STUDY_PAGES = Path("/root/selected_pages.json")

# The 18k model's validation scores from the scaling study, for comparison.
BASELINE = {"mAP50": 0.8411, "mAP50_95": 0.7478, "precision": 0.9178, "recall": 0.7444}

DATA.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
fs = HfFileSystem()
ROOT = f"datasets/{REPO}@{REVISION}"

settings.update({"wandb": False, "mlflow": False, "comet": False, "clearml": False})
assert torch.cuda.is_available(), "No GPU found."
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4090


### Progress tracker

Every long step reports through `log()`. Each line is printed here **and** added
to `progress.log`, so you can still follow the run if the browser tab
disconnects (Jupyter stops showing output, but the kernel keeps running).
From your own terminal:

```
ssh -p 54358 root@81.27.69.180 'tail -f /root/long_training/progress.log'
```

In [4]:
PROGRESS_LOG = OUTPUT / "progress.log"


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(PROGRESS_LOG, "a") as file:
        file.write(line + "\n")


def duration(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


def eta(started, done, total):
    elapsed = time.time() - started
    remaining = elapsed / max(done, 1) * (total - done)
    return f"elapsed {duration(elapsed)}, about {duration(remaining)} left"


log("Notebook started.")

[11:31:10] Notebook started.


## 4. List every page in the dataset

We read only two small columns (page metadata and annotation classes), not the
images. This tells us, for every page: where it is stored, which document it
came from, what type of document it is, and how many tables it has.

The result is cached, so this is slow only the first time.

In [5]:
def build_index(split):
    cache_path = DATA / f"index_{split}_{REVISION[:8]}.json"
    if cache_path.exists():
        log(f"Index {split}: loaded from cache.")
        return pd.read_json(cache_path, orient="records")

    files = sorted(fs.glob(f"{ROOT}/data/{split}-*.parquet"))
    log(f"Index {split}: scanning {len(files)} files...")
    started = time.time()

    pages = []
    for number, file in enumerate(files, start=1):
        # Small blocks: we read two thin columns, so big blocks would waste bandwidth.
        with fs.open(file, "rb", block_size=64 * 1024) as stream:
            parquet = pq.ParquetFile(stream, pre_buffer=False)

            for group in range(parquet.num_row_groups):
                rows = parquet.read_row_group(
                    group, columns=["metadata", "category_id"], use_threads=False
                ).to_pylist()

                for row_number, row in enumerate(rows):
                    meta = row["metadata"]
                    pages.append({
                        "file": file,
                        "group": group,
                        "row": row_number,
                        "page_hash": meta["page_hash"],
                        "domain": meta["doc_category"],
                        "document": f"{meta['collection']}/{meta['original_filename']}",
                        "tables": row["category_id"].count(TABLE_CATEGORY_ID),
                    })

        log(f"Index {split}: file {number}/{len(files)}, {len(pages):,} pages so far, "
            f"{eta(started, number, len(files))}")

    index = pd.DataFrame(pages)
    index.to_json(cache_path, orient="records")
    return index

## 5. Choose the pages

This is the **same selection code as the scaling study**, so we get the same pages.

1. **Shuffle, then keep at most 10 pages per document**, so one long report
   cannot dominate the sample.
2. **Stratify** by document type and by "has a table / has no table", so every
   kind of page is represented in proportion to its size.

In [6]:
def allocate_quotas(group_sizes, total):
    # Every group gets one place, the rest is shared in proportion to size.
    quotas = pd.Series(1, index=group_sizes.index)
    spare = group_sizes - 1
    exact_share = (total - len(group_sizes)) * spare / max(spare.sum(), 1)
    quotas = quotas + exact_share.astype(int)

    # Rounding down leaves a few places; give them to the largest remainders.
    leftover = total - quotas.sum()
    if leftover > 0:
        remainders = (exact_share - exact_share.astype(int))[quotas < group_sizes]
        quotas[remainders.nlargest(leftover).index] += 1

    assert quotas.sum() == total
    return quotas


def select_pages(index, count):
    # Step 1: shuffle, then cap pages per document.
    pool = index.sample(frac=1, random_state=SEED)
    pool = pool.groupby("document", sort=False).head(MAX_PAGES_PER_DOCUMENT)
    pool = pool.assign(has_table=pool["tables"] > 0)

    # Step 2: take a proportional share from each (document type, has table) group.
    groups = pool.groupby(["domain", "has_table"], sort=True)
    quotas = allocate_quotas(groups.size(), count)
    selected = pd.concat([pages.head(quotas[name]) for name, pages in groups])

    assert len(selected) == count and selected["page_hash"].is_unique
    return selected

In [7]:
pages = {
    "train": select_pages(build_index("train"), PAGES_TRAIN).sample(frac=1, random_state=SEED),
    "validation": select_pages(build_index("validation"), PAGES_VALIDATION),
    "test": select_pages(build_index("test"), PAGES_TEST),
}

for split, selected in pages.items():
    print(f"{split:<11} {len(selected):>6,} pages, "
          f"{(selected['tables'] > 0).mean():.0%} contain a table")

[11:31:10] Index train: loaded from cache.
[11:31:10] Index validation: loaded from cache.
[11:31:10] Index test: loaded from cache.
train       18,000 pages, 21% contain a table
validation     500 pages, 24% contain a table
test           500 pages, 23% contain a table


### Check: are these the same pages as the scaling study?

If they are, any change in score comes from training, not from different data.

In [8]:
if SCALING_STUDY_PAGES.exists():
    previous = json.loads(SCALING_STUDY_PAGES.read_text())
    for split, selected in pages.items():
        same = {r["page_hash"] for r in previous[split]} == set(selected["page_hash"])
        assert same, f"{split}: different pages from the scaling study!"
    print("Same pages as the scaling study in every split.")
else:
    print("selected_pages.json not found, skipping the comparison.")

Same pages as the scaling study in every split.


## 6. Download the pages and write YOLO labels

**Label format.** DocLayNet boxes are pixels `[left, top, width, height]`. YOLO
wants `class center_x center_y width height`, each divided by the page size.
Pages with no table get an empty label file, so the model also learns what a
table-free page looks like.

In [9]:
def to_yolo_labels(row, width, height):
    lines = []
    for category, (left, top, box_width, box_height) in zip(row["category_id"], row["bboxes"]):
        if category != TABLE_CATEGORY_ID:
            continue

        # Clip the box to the page in case an annotation runs over the edge.
        right = min(left + box_width, width)
        bottom = min(top + box_height, height)
        left, top = max(left, 0), max(top, 0)

        center_x = (left + right) / 2 / width
        center_y = (top + bottom) / 2 / height
        lines.append(f"{TABLE_CLASS_ID} {center_x:.6f} {center_y:.6f} "
                     f"{(right - left) / width:.6f} {(bottom - top) / height:.6f}")
    return lines


def download_split(selected, split):
    images_dir = DATA / "images" / split
    labels_dir = DATA / "labels" / split
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    image_paths = {}
    log(f"Download {split}: {len(selected):,} pages from {selected['file'].nunique()} files...")
    started = time.time()

    # Open each Parquet file once and read only the row groups we need.
    for file, file_pages in selected.groupby("file"):
        # Big blocks: here we read whole image columns, so fewer round trips is faster.
        with fs.open(file, "rb", block_size=8 * 1024 * 1024) as stream:
            parquet = pq.ParquetFile(stream, pre_buffer=False)

            for group, group_pages in file_pages.groupby("group"):
                block = parquet.read_row_group(
                    int(group), columns=["image", "bboxes", "category_id", "metadata"],
                    use_threads=False,
                )
                for row in block.take(group_pages["row"].tolist()).to_pylist():
                    page_hash = row["metadata"]["page_hash"]
                    raw = row["image"]["bytes"]

                    # Save the image bytes as they are: no decoding, no quality loss.
                    with Image.open(BytesIO(raw)) as image:
                        width, height = image.size
                        suffix = image.format.lower()
                    image_path = images_dir / f"{page_hash}.{suffix}"
                    image_path.write_bytes(raw)

                    labels = to_yolo_labels(row, width, height)
                    (labels_dir / f"{page_hash}.txt").write_text("\n".join(labels))

                    image_paths[page_hash] = str(image_path)

            done = len(image_paths)
            log(f"Download {split}: {done:,}/{len(selected):,} pages, "
                f"{eta(started, done, len(selected))}")

    assert len(image_paths) == len(selected), "Some pages were not downloaded."
    # Keep the order of `selected`, not the order the files were read in.
    return [image_paths[page_hash] for page_hash in selected["page_hash"]]

In [10]:
image_lists = {split: download_split(selected, split) for split, selected in pages.items()}

[11:31:11] Download train: 18,000 pages from 72 files...
[11:31:28] Download train: 199/18,000 pages, elapsed 00:00:17, about 00:25:36 left
[11:31:44] Download train: 387/18,000 pages, elapsed 00:00:33, about 00:25:43 left
[11:32:01] Download train: 556/18,000 pages, elapsed 00:00:50, about 00:26:23 left
[11:32:17] Download train: 746/18,000 pages, elapsed 00:01:06, about 00:25:37 left
[11:32:33] Download train: 948/18,000 pages, elapsed 00:01:22, about 00:24:39 left
[11:32:49] Download train: 1,131/18,000 pages, elapsed 00:01:38, about 00:24:25 left
[11:33:05] Download train: 1,335/18,000 pages, elapsed 00:01:54, about 00:23:51 left
[11:33:21] Download train: 1,526/18,000 pages, elapsed 00:02:10, about 00:23:30 left
[11:33:37] Download train: 1,709/18,000 pages, elapsed 00:02:26, about 00:23:16 left
[11:33:53] Download train: 1,877/18,000 pages, elapsed 00:02:41, about 00:23:11 left
[11:34:08] Download train: 2,007/18,000 pages, elapsed 00:02:57, about 00:23:34 left
[11:34:24] Downloa

## 7. Sanity check: no document appears in two splits

If one document had pages in both train and validation, the model could score
well by memorising that document's layout.

In [11]:
documents = {split: set(selected["document"]) for split, selected in pages.items()}
assert documents["train"].isdisjoint(documents["validation"])
assert documents["train"].isdisjoint(documents["test"])
assert documents["validation"].isdisjoint(documents["test"])
print("No document is shared between splits.")

No document is shared between splits.


## 8. Write the dataset config

In [12]:
for split, paths in image_lists.items():
    (DATA / f"{split}.txt").write_text("\n".join(paths))

data_config = DATA / "data.yaml"
data_config.write_text(yaml.safe_dump({
    "path": str(DATA),
    "train": "train.txt",
    "val": "validation.txt",
    "test": "test.txt",
    "names": {TABLE_CLASS_ID: "table"},
}))
print(data_config.read_text())

names:
  0: table
path: /root/table_data
test: test.txt
train: train.txt
val: validation.txt



## 9. Train

- `epochs=60, patience=10`: train for up to 60 epochs, and stop early if
  validation mAP has not improved for 10 epochs in a row.
- `imgsz=1024`: pages go into the network at ~native resolution.
- `mosaic=0, fliplr=0, flipud=0`: these augmentations stitch pages together or
  mirror them. Real documents never look like that, so they are switched off.

Ultralytics saves the best epoch's weights as `best.pt` automatically.

Ultralytics shows its own progress bar for each epoch. On top of that, the
function below writes one summary line per epoch to the progress log: the
scores, how long is left, and how close early stopping is.

In [13]:
reported_epochs = set()


def report_epoch(trainer):
    epoch = trainer.epoch + 1
    if epoch in reported_epochs:  # Ultralytics calls this again after its final evaluation.
        return
    reported_epochs.add(epoch)

    metrics = trainer.metrics
    epochs_without_gain = epoch - trainer.stopper.best_epoch
    log(f"Epoch {epoch}/{EPOCHS}: "
        f"mAP50 {metrics['metrics/mAP50(B)']:.3f}, "
        f"mAP50-95 {metrics['metrics/mAP50-95(B)']:.3f}, "
        f"val box loss {metrics['val/box_loss']:.3f} | "
        f"best epoch {trainer.stopper.best_epoch}, "
        f"no improvement for {epochs_without_gain}/{PATIENCE} | "
        f"{eta(training_started, epoch, EPOCHS)} (if it runs all {EPOCHS})")


model = YOLO(PRETRAINED)
model.add_callback("on_fit_epoch_end", report_epoch)
training_started = time.time()
log(f"Training started: {PAGES_TRAIN:,} pages, {IMGSZ} px, up to {EPOCHS} epochs.")

model.train(
    data=str(data_config),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    device=0,
    workers=WORKERS,
    mosaic=0.0,
    fliplr=0.0,
    flipud=0.0,
    project=str(OUTPUT),
    name="train",
    exist_ok=True,
)

run_dir = Path(model.trainer.save_dir)
best_weights = OUTPUT / "best_1024_60ep.pt"
shutil.copy2(run_dir / "weights" / "best.pt", best_weights)
log(f"Training finished in {duration(time.time() - training_started)}. "
    f"Best weights saved to {best_weights}")

[11:51:20] Training started: 18,000 pages, 1024 px, up to 60 epochs.
New https://pypi.org/project/ultralytics/8.4.190 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/table_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=

## 10. Did it converge this time?

The whole point of this run. If the curves have flattened before the last epoch,
the model has learned what it can from this data at this size.

In [14]:
history = pd.read_csv(run_dir / "results.csv")
history.columns = history.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["metrics/mAP50(B)"], label="mAP50")
axes[0].plot(history["epoch"], history["metrics/mAP50-95(B)"], label="mAP50-95")
axes[0].set_title("Validation mAP per epoch")
axes[1].plot(history["epoch"], history["train/box_loss"], label="train box loss")
axes[1].plot(history["epoch"], history["val/box_loss"], label="validation box loss")
axes[1].set_title("Box loss per epoch")
for axis in axes:
    axis.set_xlabel("epoch")
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
fig.savefig(OUTPUT / "training_curves.png", dpi=150)
plt.show()

best_epoch = history.loc[history["metrics/mAP50-95(B)"].idxmax(), "epoch"]
print(f"Ran {len(history)} of {EPOCHS} epochs. Best epoch: {best_epoch}.")

<Figure size 1200x400 with 2 Axes>

Ran 60 of 60 epochs. Best epoch: 56.


## 11. Score on validation and compare with the scaling study

Same validation pages and same scoring settings as before.

In [15]:
def evaluate(weights, split):
    metrics = YOLO(str(weights)).val(
        data=str(data_config),
        split=split,
        imgsz=IMGSZ,
        batch=4,
        device=0,
        half=False,
        rect=False,
        conf=0.001,  # Keep low-confidence boxes so the PR curve is complete.
        project=str(OUTPUT),
        name=split,
        exist_ok=True,
    )
    return {
        "mAP50": round(float(metrics.box.map50), 4),
        "mAP50_95": round(float(metrics.box.map), 4),
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
    }


validation_scores = evaluate(best_weights, "val")
(OUTPUT / "validation_results.json").write_text(json.dumps(validation_scores, indent=2))

comparison = pd.DataFrame({
    "18k, 640px, 20 epochs": BASELINE,
    "18k, 1024px, 60 epochs": validation_scores,
})
comparison["change"] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
comparison.round(4)

Ultralytics 8.3.221 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3607.7±213.0 MB/s, size: 337.7 KB)
val: Scanning /root/table_data/labels/validation.cache... 500 images, 382 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 792.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 125/125 38.0it/s 3.3s0.1s
                   all        500        180      0.934      0.806      0.884      0.809
Speed: 2.2ms preprocess, 2.4ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /root/long_training/val


,"18k, 640px, 20 epochs","18k, 1024px, 60 epochs",change
mAP50,0.8411,0.8836,0.0425
mAP50_95,0.7478,0.8092,0.0614
precision,0.9178,0.9335,0.0157
recall,0.7444,0.8056,0.0612


## 12. Final test score (run once, at the very end)

Validation was used to choose the model, so it is slightly optimistic. The test
split gives the honest number. Turn this on only once you have decided this is
the model you will report.

For the report: the test split was already used once, to score the 18k model
from the scaling study (mAP50 0.893, mAP50-95 0.817).

In [17]:
RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    test_scores = evaluate(best_weights, "test")
    (OUTPUT / "test_results.json").write_text(json.dumps(test_scores, indent=2))
    print(json.dumps(test_scores, indent=2))
else:
    print("Skipped. Set RUN_FINAL_TEST = True when the model is final.")

Ultralytics 8.3.221 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3026.7±710.8 MB/s, size: 330.8 KB)
val: Scanning /root/table_data/labels/test.cache... 500 images, 384 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.4Mit/s 0.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 125/125 37.8it/s 3.3s0.1s
                   all        500        200      0.953       0.85      0.918      0.863
Speed: 2.2ms preprocess, 2.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /root/long_training/test
{
  "mAP50": 0.918,
  "mAP50_95": 0.863,
  "precision": 0.953,
  "recall": 0.85
}


## 13. What to download

Everything is in `OUTPUT` (`/root/long_training`):

| File | What it is |
|---|---|
| `best_1024_60ep.pt` | The trained model |
| `validation_results.json` | Validation scores |
| `training_curves.png` | mAP and loss per epoch: did it converge? |
| `train/` | Ultralytics plots: PR curves, confusion matrix, sample batches |
| `val/` | Validation plots |
| `test_results.json`, `test/` | Only if the final test was run |